# Par change

**Table of contents**<a id='toc0_'></a>    
- 1. [Imports](#toc1_)    
- 2. [Pre-train model](#toc2_)    
- 3. [Load pre-trained model and train again following parameter change](#toc3_)    

<!-- vscode-jupyter-toc-config
    numbering=true
    anchor=true
    flat=false
    minLevel=2
    maxLevel=6
    /vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## 1. <a id='toc1_'></a>[Imports](#toc0_)

In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE' # without this python may crash when plotting from matplotlib

import numpy as np
import time
import matplotlib.pyplot as plt
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':'0.25','grid.linestyle': '--'})
plt.rcParams.update({'font.size':14})
plt.rcParams.update({'font.family':'serif'})

In [ ]:
from EconDLSolvers import choose_gpu
from NonConvexDurablesModel import NonConvexDurablesModelClass
from NonConvexDurablesModelVFI import NonConvexDurablesModelVFIClass

In [4]:
os.makedirs('../output', exist_ok=True)

## 2. <a id='toc2_'></a>[Pre-train model](#toc0_)

In [5]:
device = choose_gpu()

do_train = False
if do_train:
    K_time = 1
    full = False

    par = {'D':1, 'full':False}
    train  = {'K_time':K_time}
    model = NonConvexDurablesModelClass(device=device,algoname='DeepVPDDC',train=train,par=par)
    model.solve(do_print=True)
    model.save('../output/NonConvexDurablesModel_DL_1D_sigmoidpretrain.pt')
else:
    path = '../output/NonConvexDurablesModel_DL_1D_sigmoid.pt' # path for final model solution
    model = NonConvexDurablesModelClass(device=device,load=path)

No GPU available, using CPU


## 3. <a id='toc3_'></a>[Load pre-trained model and train again following parameter change](#toc0_)

In [ ]:
# load previous solution again
model_new = NonConvexDurablesModelClass(device=device,load=path)

# make model keep training
del model_new.info['solve_in_progress']

print(model_new.par.beta)
model_new.par.beta = 0.97
model_new.train.K_time = 60.0 # change duration of new training

0.965


In [ ]:
model_new.solve(do_print=True)
model_new.more_simulation_outcomes()
model_new.euler_errors_DL()

started solving: 2026-02-09 15:26:15
k =     0 of inf: sim.R =    -5.99684238 [best:    -5.99684238] [0.6 secs] [value_epochs =  88] [policy_epochs =   0] [  0.01 mins] [policy_lr = 9.9e-04] [value_lr = 9.8e-04]
k =    10 of inf: sim.R =    -5.99674845 [best:    -5.99674845] [0.8 secs] [value_epochs = 150] [policy_epochs =   0] [  0.07 mins] [policy_lr = 9.9e-04] [value_lr = 9.8e-04]
k =    20 of inf: sim.R =    -5.99688911 [best:    -5.99674845] [0.1 secs] [value_epochs =  33] [policy_epochs =   0] [  0.11 mins] [policy_lr = 9.9e-04] [value_lr = 9.8e-04] no improvements in 0.0 mins
k =    30 of inf: sim.R =    -5.99682045 [best:    -5.99674845] [0.1 secs] [value_epochs =  40] [policy_epochs =   0] [  0.13 mins] [policy_lr = 9.9e-04] [value_lr = 9.8e-04] no improvements in 0.1 mins
k =    40 of inf: sim.R =    -5.99681425 [best:    -5.99674845] [0.1 secs] [value_epochs =  36] [policy_epochs =   0] [  0.14 mins] [policy_lr = 9.9e-04] [value_lr = 9.8e-04] no improvements in 0.1 mins
k = 

In [ ]:
model_new.save('../output/NonConvexDurablesModel_DL_1D_sigmoidaltbeta.pt')

## Comparison with DP

In [ ]:
model_DP = NonConvexDurablesModelVFIClass(par={'D':1,'full':True})
model_DP.par.beta = model_new.par.beta
model_DP.link_to_cpp()


In [ ]:
t0_solve = time.perf_counter()
model_DP.solve()
t1_solve = time.perf_counter()

model_DP.info['time'] = t1_solve - t0_solve

In [ ]:
model_DP.simulate_R()
model_DP.compute_euler_error()
model_DP.compute_transfer_func()

In [ ]:
print(model_DP.sim.R)

In [ ]:
vfi = model_DP.vfi

# keeper
vfi.sol_sav_share_keep = None
vfi.sol_v_keep = None
vfi.sol_func_evals_keep = None
vfi.sol_flag_keep = None

# adjust durable 1
vfi.sol_exp_share_adj1 = None
vfi.sol_c_share_adj1 = None
vfi.sol_v_adj1 = None
vfi.sol_func_evals_adj1 = None
vfi.sol_flag_adj1 = None

# adjust durable 2
vfi.sol_exp_share_adj2 = None
vfi.sol_c_share_adj2 = None
vfi.sol_v_adj2 = None
vfi.sol_func_evals_adj2 = None
vfi.sol_flag_adj2 = None

# adjust both durables
vfi.sol_exp_share_adj12 = None
vfi.sol_c_share_adj12 = None
vfi.sol_d1_share_adj12 = None
vfi.sol_v_adj12 = None
vfi.sol_func_evals_adj12 = None
vfi.sol_flag_adj12 = None

# post-decision
vfi.sol_w = None

vfi.sol_exp_share_adj = None
vfi.sol_c_share_adj = None
vfi.sol_v_adj = None
vfi.sol_func_evals_adj = None
vfi.sol_flag_adj = None

In [ ]:
model_DP.save('../output/NonConvexDurablesModel_DP_1D_altbeta.pt')

In [ ]:
model_DP.cpp.delink()

In [ ]:
del model_DP
del vfi